In [1]:
# ==========================================
# ENVIRONMENT SETUP & INSTALLATIONS
# ==========================================
# Install and import libraries

%pip install geopandas fiona pyarrow fastparquet -q

import pandas as pd
import geopandas as gpd
import fiona
import os
import gc
import json

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 948.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 33.8 MB/s eta 0:00:00


In [ ]:
# ==========================================
# LOCATE THE PROJECT FOLDER
# ==========================================
# Finds the cloned repository (the folder that contains 03_scripts).
# Works locally, in Colab and elsewhere. Set the PROJECT_ROOT environment variable to override.
import os
from pathlib import Path

def find_project_root():
    if os.environ.get("PROJECT_ROOT"):
        return Path(os.environ["PROJECT_ROOT"])
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "03_scripts").is_dir():
            return folder
    colab_clones = sorted(Path("/content").glob("*/03_scripts"))
    if colab_clones:
        return colab_clones[0].parent
    raise FileNotFoundError("Project folder not found. Run the notebook from inside the cloned repository, or set PROJECT_ROOT.")

PROJECT_ROOT = str(find_project_root())
print(f"Project folder: {PROJECT_ROOT}")

In [3]:
# ==========================================
# PART 1 (RESIDENTIAL) PRE-CHECK: DATA CHECK
# ==========================================

import geopandas as gpd
import pandas as pd
import os
import fiona

print("Executing Script 3 - Phase 1 Pre-Check...\n")

# Define Paths
BASE_DIR = PROJECT_ROOT
input_res = os.path.join(BASE_DIR, '02_processed_data/02_spatial/residential_geocoded.parquet')
lsoa_geojson_path = os.path.join(BASE_DIR, '01_raw_data/spatial/LSOA_2021/LSOA_2021.geojson')

# Updated IMD Path
imd_csv_path = os.path.join(BASE_DIR, '01_raw_data/spatial/English_IMD/English_IMD.csv')

# 1. Check Residential Data
print("--- 1. RESIDENTIAL SPATIAL DATA ---")
try:
    gdf_res = gpd.read_parquet(input_res)
    print(f"Loaded: {len(gdf_res):,} properties.")
    print(f"Active CRS: {gdf_res.crs.name if gdf_res.crs else 'None'}")
    print(f"Geometry Type: {gdf_res.geometry.geom_type.iloc[0]}")
except Exception as e:
    print(f"Error loading residential file: {e}")

# 2. Check LSOA Boundaries (Memory-Safe Peek)
print("\n--- 2. LSOA 2021 BOUNDARIES ---")
try:
    with fiona.open(lsoa_geojson_path) as src:
        print(f"Total National LSOAs in file: {len(src):,}")
        print(f"LSOA CRS: {src.crs}")
        print(f"Available Columns: {list(src.schema['properties'].keys())[:6]}...")
except Exception as e:
    print(f"Error reading LSOA file: {e}")

# 3. Check IMD Tabular Data
print("\n--- 3. IMD 2025 TABULAR DATA ---")
if os.path.exists(imd_csv_path):
    df_imd = pd.read_csv(imd_csv_path, nrows=5)
    print("Successfully located IMD file. Columns found:")
    for col in df_imd.columns:
        print(f" - {col}")
else:
    print(f"WARNING: Could not find IMD file at {imd_csv_path}.")

Executing Script 3 - Phase 1 Pre-Check...

--- 1. RESIDENTIAL SPATIAL DATA ---
Loaded: 43,413 properties.
Active CRS: OSGB36 / British National Grid
Geometry Type: Point

--- 2. LSOA 2021 BOUNDARIES ---
Total National LSOAs in file: 35,672
LSOA CRS: EPSG:4326
Available Columns: ['FID', 'LSOA21CD', 'LSOA21NM', 'LSOA21NMW', 'BNG_E', 'BNG_N']...

--- 3. IMD 2025 TABULAR DATA ---
Successfully located IMD file. Columns found:
 - LSOA code (2021)
 - LSOA name (2021)
 - Local Authority District code (2024)
 - Local Authority District name (2024)
 - Index of Multiple Deprivation (IMD) Score
 - Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)
 - Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)
 - Income Score (rate)
 - Income Rank (where 1 is most deprived)
 - Income Decile (where 1 is most deprived 10% of LSOAs)
 - Employment Score (rate)
 - Employment Rank (where 1 is most deprived)
 - Employment Decile (where 1 is most deprived 10% of LSOA

In [4]:
# ==========================================
# PART 1 (RESIDENTIAL) - LSOA AND IMD FULL EXECUTION (OVERLAY + MERGE)
# ==========================================
import pandas as pd
import geopandas as gpd
import os
import gc
import json

print("Initiating full Part 1 execution (Residential Overlay + Merge + Log)...\n")

# Define paths
BASE_DIR = PROJECT_ROOT
input_res = os.path.join(BASE_DIR, '02_processed_data/02_spatial/residential_geocoded.parquet')
buffer_path = os.path.join(BASE_DIR, '01_raw_data/Stations_and_2000m_Buffer/All_Stations_2000m_Buffer.geojson')
lsoa_path = os.path.join(BASE_DIR, '01_raw_data/spatial/LSOA_2021/LSOA_2021.geojson')
imd_path = os.path.join(BASE_DIR, '01_raw_data/spatial/English_IMD/English_IMD.csv')
output_res = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/devon_master_residential.parquet')
log_path = os.path.join(BASE_DIR, '05_logs/script_logs/attrition_log.json')

# 1. Overlay Phase
gdf_res = gpd.read_parquet(input_res)
initial_res_count = len(gdf_res)
gdf_buffer = gpd.read_file(buffer_path).to_crs("EPSG:27700")
bbox_4326 = tuple(gdf_buffer.to_crs("EPSG:4326").total_bounds)
gdf_lsoa = gpd.read_file(lsoa_path, bbox=bbox_4326).to_crs("EPSG:27700")[['LSOA21CD', 'geometry']]

gdf_res['centroid'] = gdf_res.geometry.centroid
gdf_res = gdf_res.set_geometry('centroid')
gdf_res = gpd.sjoin(gdf_res, gdf_lsoa, how='left', predicate='within').drop(columns=['centroid', 'index_right'])
gdf_res = gdf_res.set_geometry('geometry')

lsoa_match_count = gdf_res['LSOA21CD'].notna().sum()
dropped_spatial = initial_res_count - lsoa_match_count
print(f"Spatial Overlay: Attached LSOA codes to {lsoa_match_count:,} properties ({dropped_spatial:,} dropped).")

# 2. Merge Phase
df_imd = pd.read_csv(imd_path)
gdf_res = gdf_res.merge(
    df_imd[["LSOA code (2021)", "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)"]],
    left_on='LSOA21CD',
    right_on="LSOA code (2021)",
    how='left'
).rename(columns={"Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": 'IMD_Decile'}).drop(columns=["LSOA code (2021)"])

final_match_count = gdf_res['IMD_Decile'].notna().sum()
dropped_tabular = lsoa_match_count - final_match_count
print(f"Tabular Merge: Harmonised {final_match_count:,} properties with IMD deciles ({dropped_tabular:,} dropped).")

# 3. Cleanup & Save
cols_to_drop = ['Match_String_1', 'Match_String_Swapped']
gdf_res = gdf_res.drop(columns=[col for col in cols_to_drop if col in gdf_res.columns])
os.makedirs(os.path.dirname(output_res), exist_ok=True)
gdf_res.to_parquet(output_res, index=False)

# 4. Update Log
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        attrition_log = json.load(f)
else:
    attrition_log = {}

# Add new Part 1 keys
attrition_log["Part_1_Residential_Spatial_Overlay"] = {
    "01_Input_Properties": initial_res_count,
    "02_LSOA_Matched": int(lsoa_match_count),
    "03_Properties_Dropped": int(dropped_spatial),
    "Note": "Spatial overlay via centroid-to-polygon."
}

attrition_log["Part_1_Residential_IMD_Merge"] = {
    "01_Input_Properties": int(lsoa_match_count),
    "02_IMD_Decile_Matched": int(final_match_count),
    "03_Properties_Dropped": int(dropped_tabular),
    "04_Final_Retention_Rate": round((final_match_count / initial_res_count) * 100, 2),
    "Note": "Tabular merge with English IMD 2025."
}

with open(log_path, 'w') as f:
    json.dump(attrition_log, f, indent=4)

print("\nPart 1 complete. Attrition log updated and file saved to 03_model_ready.")
del gdf_res, df_imd, gdf_lsoa, gdf_buffer
_ = gc.collect()

Initiating full Part 1 execution (Residential Overlay + Merge + Log)...

Spatial Overlay: Attached LSOA codes to 43,165 properties (248 dropped).
Tabular Merge: Harmonised 43,165 properties with IMD deciles (0 dropped).

Part 1 complete. Attrition log updated and file saved to 03_model_ready.


In [5]:
# ==========================================
# PART 1 (RESIDENTIAL) - POST-CHECK: LSOA & IMD VERIFICATION
# ==========================================
import geopandas as gpd
import os

print("Executing Phase 1 Post-Merge Diagnostic Check...\n")

# Define path to the newly saved model-ready file
BASE_DIR = PROJECT_ROOT
output_res = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/devon_master_residential.parquet')

# Load the dataset
gdf_check = gpd.read_parquet(output_res)

# erify dataset dimensions
print("--- Dataset Dimension ---")
print(f"Total Dimensions: {gdf_check.shape[0]:,} rows and {gdf_check.shape[1]} columns.")

# Check new socio-economic columns
print("\n--- LSOA and IMD ---")
print("Missing Values in Target Columns:")
print(gdf_check[['LSOA21CD', 'IMD_Decile']].isna().sum())

# Verify spatial quality
print(f"Active CRS: {gdf_check.crs.name if gdf_check.crs else 'None'}")
print("Geometry Types:")
print(gdf_check.geometry.geom_type.value_counts())

# Display sample
print("\n--- Data Sample ---")
# Display the critical econometric variables to ensure they look mathematically sound
print(gdf_check[['Transaction_ID', 'Postcode', 'LSOA21CD', 'IMD_Decile']].head())

print("\nVerification Complete: Residential dataset is perfectly harmonised and ready for routing!")

Executing Phase 1 Post-Merge Diagnostic Check...

--- Dataset Dimension ---
Total Dimensions: 43,413 rows and 20 columns.

--- LSOA and IMD ---
Missing Values in Target Columns:
LSOA21CD      248
IMD_Decile    248
dtype: int64
Active CRS: OSGB36 / British National Grid
Geometry Types:
Point    43413
Name: count, dtype: int64

--- Data Sample ---
                           Transaction_ID  Postcode   LSOA21CD  IMD_Decile
0  {4C1B673F-0261-D8EA-E063-4704A8C0BF7D}   TQ2 7RS  E01015250         6.0
1  {4C1B673F-0263-D8EA-E063-4704A8C0BF7D}  EX17 2BP  E01020060         4.0
2  {4C1B673F-027D-D8EA-E063-4704A8C0BF7D}   EX4 1SR  E01019984         8.0
3  {4C1B673F-0293-D8EA-E063-4704A8C0BF7D}   PL1 5BX  E01015151         1.0
4  {4C1B673E-FDEB-D8EA-E063-4704A8C0BF7D}  EX31 3RH  E01020125         9.0

Verification Complete: Residential dataset is perfectly harmonised and ready for routing!


In [6]:
# ==========================================
# PART 1 (RESIDENTIAL) POST-CHECK: QUERY RESIDENTIAL MASTER PARQUET (ALL COLUMNS)
# ==========================================
import geopandas as gpd
import pandas as pd
import os

print("Querying the final devon_master_residential.parquet file...\n")

# Define the path to the model-ready dataset
BASE_DIR = PROJECT_ROOT
file_path = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/devon_master_residential.parquet')

if os.path.exists(file_path):
    # Load the GeoParquet file
    gdf_master = gpd.read_parquet(file_path)

    print("--- 1. DATASET INFO ---")
    print(f"Total Rows: {len(gdf_master):,}")
    print(f"Total Columns: {len(gdf_master.columns)}")
    print(f"Active CRS: {gdf_master.crs.name if gdf_master.crs else 'None'}\n")

    print("--- 2. ALL COLUMNS & DATA TYPES ---")
    print(gdf_master.dtypes)

    print("\n--- 3. DATA SAMPLE (FIRST 5 ROWS, ALL COLUMNS) ---")
    # Temporarily tell Pandas to display every single column without truncation
    with pd.option_context('display.max_columns', None):
        print(gdf_master.head())

else:
    print(f"File not found at: {file_path}")

Querying the final devon_master_residential.parquet file...

--- 1. DATASET INFO ---
Total Rows: 43,413
Total Columns: 20
Active CRS: OSGB36 / British National Grid

--- 2. ALL COLUMNS & DATA TYPES ---
Transaction_ID              object
Price                        int64
Price_Normalised           float64
Log_Price                  float64
Date_of_Transfer    datetime64[ns]
Postcode                    object
Property_Type               object
Is_New_Build                 int64
PAON                        object
SAON                        object
Street                      object
Total_Floor_Area           float64
Log_Floor_Area             float64
UPRN                        object
EPC_Rating                  object
east1m                     float64
north1m                    float64
geometry                  geometry
LSOA21CD                    object
IMD_Decile                 float64
dtype: object

--- 3. DATA SAMPLE (FIRST 5 ROWS, ALL COLUMNS) ---
                           Trans

In [7]:
# ==========================================
# PART 2 (COMMERCIAL) PRE-CHECK: DATA CHECK
# ==========================================
import geopandas as gpd
import pandas as pd
import os
import fiona

print("Executing Script 3 - Phase 2 Pre-Check...\n")

# Define paths
BASE_DIR = PROJECT_ROOT
input_com = os.path.join(BASE_DIR, '02_processed_data/02_spatial/commercial_geocoded.parquet')
lsoa_geojson_path = os.path.join(BASE_DIR, '01_raw_data/spatial/LSOA_2021/LSOA_2021.geojson')
imd_csv_path = os.path.join(BASE_DIR, '01_raw_data/spatial/English_IMD/English_IMD.csv')

# 1. Check commercial spatial data
try:
    gdf_com = gpd.read_parquet(input_com)
    print(f"Loaded: {len(gdf_com):,} commercial properties.")
    print(f"Active CRS: {gdf_com.crs.name if gdf_com.crs else 'None'}")
    print(f"Geometry Type: {gdf_com.geometry.geom_type.iloc[0]}")
except Exception as e:
    print(f"Error loading commercial file: {e}")

# 2. Check LSOA boundaries
print("\n--- 2. LSOA 2021 BOUNDARIES ---")
try:
    with fiona.open(lsoa_geojson_path) as src:
        print(f"Total national LSOAs: {len(src):,}")
        print(f"LSOA CRS: {src.crs}")
except Exception as e:
    print(f"Error reading LSOA file: {e}")

# 3. Check IMD tabular data
print("\n--- 3. IMD 2025 TABULAR DATA ---")
if os.path.exists(imd_csv_path):
    df_imd = pd.read_csv(imd_csv_path, nrows=2)
    print("Successfully located IMD file. Join column found:")
    print(f" - {'LSOA code (2021)' in df_imd.columns}")
else:
    print(f"WARNING: Could not find IMD file at {imd_csv_path}.")

Executing Script 3 - Phase 2 Pre-Check...

Loaded: 28,361 commercial properties.
Active CRS: OSGB36 / British National Grid
Geometry Type: Point

--- 2. LSOA 2021 BOUNDARIES ---
Total national LSOAs: 35,672
LSOA CRS: EPSG:4326

--- 3. IMD 2025 TABULAR DATA ---
Successfully located IMD file. Join column found:
 - True


In [8]:
# ==========================================
# PART 2 (COMMERCIAL) - LSOA AND IMD FULL EXECUTION (OVERLAY + MERGE)
# ==========================================
import pandas as pd
import geopandas as gpd
import os
import gc
import json

print("Initiating full Part 2 execution (Commercial Overlay + Merge + Log)...\n")

# Define paths
BASE_DIR = PROJECT_ROOT
input_com = os.path.join(BASE_DIR, '02_processed_data/02_spatial/commercial_geocoded.parquet')
buffer_path = os.path.join(BASE_DIR, '01_raw_data/Stations_and_2000m_Buffer/All_Stations_2000m_Buffer.geojson')
lsoa_path = os.path.join(BASE_DIR, '01_raw_data/spatial/LSOA_2021/LSOA_2021.geojson')
imd_path = os.path.join(BASE_DIR, '01_raw_data/spatial/English_IMD/English_IMD.csv')
output_com = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/devon_master_commercial.parquet')
log_path = os.path.join(BASE_DIR, '05_logs/script_logs/attrition_log.json')

# 1. Overlay Phase
# Commercial data is Point geometry; direct sjoin is efficient
gdf_com = gpd.read_parquet(input_com)
initial_com_count = len(gdf_com)

gdf_buffer = gpd.read_file(buffer_path).to_crs("EPSG:27700")
bbox_4326 = tuple(gdf_buffer.to_crs("EPSG:4326").total_bounds)
gdf_lsoa = gpd.read_file(lsoa_path, bbox=bbox_4326).to_crs("EPSG:27700")[['LSOA21CD', 'geometry']]

gdf_com = gpd.sjoin(gdf_com, gdf_lsoa, how='left', predicate='within').drop(columns=['index_right'])

lsoa_match_count = gdf_com['LSOA21CD'].notna().sum()
dropped_spatial = initial_com_count - lsoa_match_count
print(f"Spatial Overlay: Attached LSOA codes to {lsoa_match_count:,} commercial properties ({dropped_spatial:,} dropped).")

# 2. Merge Phase
df_imd = pd.read_csv(imd_path)
gdf_com = gdf_com.merge(
    df_imd[["LSOA code (2021)", "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)"]],
    left_on='LSOA21CD',
    right_on="LSOA code (2021)",
    how='left'
).rename(columns={"Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": 'IMD_Decile'}).drop(columns=["LSOA code (2021)"])

final_match_count = gdf_com['IMD_Decile'].notna().sum()
dropped_tabular = lsoa_match_count - final_match_count
print(f"Tabular Merge: Harmonised {final_match_count:,} commercial properties with IMD deciles ({dropped_tabular:,} dropped).")

# 3. Save & Log
os.makedirs(os.path.dirname(output_com), exist_ok=True)
gdf_com.to_parquet(output_com, index=False)

# Update log
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        attrition_log = json.load(f)
else:
    attrition_log = {}

# Update attrition log
attrition_log["Part_2_Commercial_Foundation"] = {
    "01_Input_Properties": initial_com_count,
    "02_LSOA_Matched": int(lsoa_match_count),
    "03_IMD_Decile_Matched": int(final_match_count),
    "04_Overall_Retention_Rate": round((final_match_count / initial_com_count) * 100, 2),
    "Note": "Combined spatial overlay and tabular merge. LSOA boundaries matched via point-in-polygon; IMD deciles merged via LSOA21CD."
}

with open(log_path, 'w') as f:
    json.dump(attrition_log, f, indent=4)

print("\nPart 2 complete. Full commercial attrition details saved to log and file written.")
del gdf_com, df_imd, gdf_lsoa, gdf_buffer
_ = gc.collect()

Initiating full Part 2 execution (Commercial Overlay + Merge + Log)...

Spatial Overlay: Attached LSOA codes to 28,169 commercial properties (192 dropped).
Tabular Merge: Harmonised 28,169 commercial properties with IMD deciles (0 dropped).

Part 2 complete. Full commercial attrition details saved to log and file written.


In [9]:
# ==========================================
# PART 2 (COMMERCIAL) - POST-CHECK: LSOA & IMD VERIFICATION
# ==========================================
import geopandas as gpd
import os
import pandas as pd

# Define path to the newly saved model-ready commercial file
BASE_DIR = PROJECT_ROOT
output_com = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/devon_master_commercial.parquet')

# Load the dataset
gdf_check = gpd.read_parquet(output_com)

# Verify dataset dimensions
print("--- Dataset Dimensions ---")
print(f"Total Dimensions: {gdf_check.shape[0]:,} rows and {gdf_check.shape[1]} columns.")

# Check socio-economic attachment
print("\n--- LSOA and IMD---")
print("Missing Values in Target Columns:")
print(gdf_check[['LSOA21CD', 'IMD_Decile']].isna().sum())

# Verify spatial quality
print("\n--- Spatial Quality ---")
print(f"Active CRS: {gdf_check.crs.name if gdf_check.crs else 'None'}")
print("Geometry Types:")
print(gdf_check.geometry.geom_type.value_counts())

# 4. Display sample
print("\n--- Data Sample ---")
# Display critical econometric variables to ensure they look mathematically sound
print(gdf_check[['LSOA21CD', 'IMD_Decile']].head())

print("\nVerification Complete: Commercial dataset is perfectly harmonised and ready for routing!")

--- Dataset Dimensions ---
Total Dimensions: 28,361 rows and 13 columns.

--- LSOA and IMD---
Missing Values in Target Columns:
LSOA21CD      192
IMD_Decile    192
dtype: int64

--- Spatial Quality ---
Active CRS: OSGB36 / British National Grid
Geometry Types:
Point    28361
Name: count, dtype: int64

--- Data Sample ---
    LSOA21CD  IMD_Decile
0  E01018791         5.0
1  E01018791         5.0
2  E01018791         5.0
3  E01018793         5.0
4  E01018798         6.0

Verification Complete: Commercial dataset is perfectly harmonised and ready for routing!


In [10]:
# ==========================================
# POST_CHECK: QUERY COMMERCIAL MASTER PARQUET (ALL COLUMNS)
# ==========================================
import geopandas as gpd
import pandas as pd
import os

print("Querying the final devon_master_commercial.parquet file...\n")

# Define the path to the model-ready commercial dataset
BASE_DIR = PROJECT_ROOT
file_path = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/devon_master_commercial.parquet')

if os.path.exists(file_path):
    # Load the GeoParquet file
    gdf_master = gpd.read_parquet(file_path)

    print("--- Dataset Info ---")
    print(f"Total Rows: {len(gdf_master):,}")
    print(f"Total Columns: {len(gdf_master.columns)}")
    print(f"Active CRS: {gdf_master.crs.name if gdf_master.crs else 'None'}\n")

    print("--- All Columns and Data Types ---")
    print(gdf_master.dtypes)

    print("\n--- Data Sample ---")
    # Temporarily tell Pandas to display every single column without truncation
    with pd.option_context('display.max_columns', None):
        print(gdf_master.head())

else:
    print(f"File not found at: {file_path}")

Querying the final devon_master_commercial.parquet file...

--- Dataset Info ---
Total Rows: 28,361
Total Columns: 13
Active CRS: OSGB36 / British National Grid

--- All Columns and Data Types ---
Assessment_UARN         object
Postcode                object
Property_Sector         object
Rateable_Value           int64
Total_Floor_Area       float64
Year                     int64
Log_Rateable_Value     float64
Log_Floor_Area         float64
east1m                 float64
north1m                float64
geometry              geometry
LSOA21CD                object
IMD_Decile             float64
dtype: object

--- Data Sample ---
  Assessment_UARN  Postcode                          Property_Sector  \
0     23492197000  PL12 4EA  SELF CATERING HOLIDAY UNIT AND PREMISES   
1     16924750000  PL12 4EB                                 CAR PARK   
2     18540093000  PL12 4EB                       STORE AND PREMISES   
3     16924543000  PL12 6AN                                 CAR PARK   
4    

In [11]:
# ==========================================
# POST_CHECK: QUERY COMMERCIAL MASTER PARQUET (ALL COLUMNS)
# ==========================================
import geopandas as gpd
import pandas as pd
import os

print("Querying the final devon_master_commercial.parquet file...\n")

# Define the path to the model-ready dataset
BASE_DIR = PROJECT_ROOT
file_path = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/devon_master_commercial.parquet')

if os.path.exists(file_path):
    # Load the GeoParquet file
    gdf_master = gpd.read_parquet(file_path)

    print("--- 1. DATASET INFO ---")
    print(f"Total Rows: {len(gdf_master):,}")
    print(f"Total Columns: {len(gdf_master.columns)}")
    print(f"Active CRS: {gdf_master.crs.name if gdf_master.crs else 'None'}\n")

    print("--- 2. ALL COLUMNS & DATA TYPES ---")
    print(gdf_master.dtypes)

    print("\n--- 3. DATA SAMPLE (FIRST 5 ROWS, ALL COLUMNS) ---")
    # Temporarily tell Pandas to display every single column without truncation
    with pd.option_context('display.max_columns', None):
        print(gdf_master.head())

else:
    print(f"File not found at: {file_path}")

Querying the final devon_master_commercial.parquet file...

--- 1. DATASET INFO ---
Total Rows: 28,361
Total Columns: 13
Active CRS: OSGB36 / British National Grid

--- 2. ALL COLUMNS & DATA TYPES ---
Assessment_UARN         object
Postcode                object
Property_Sector         object
Rateable_Value           int64
Total_Floor_Area       float64
Year                     int64
Log_Rateable_Value     float64
Log_Floor_Area         float64
east1m                 float64
north1m                float64
geometry              geometry
LSOA21CD                object
IMD_Decile             float64
dtype: object

--- 3. DATA SAMPLE (FIRST 5 ROWS, ALL COLUMNS) ---
  Assessment_UARN  Postcode                          Property_Sector  \
0     23492197000  PL12 4EA  SELF CATERING HOLIDAY UNIT AND PREMISES   
1     16924750000  PL12 4EB                                 CAR PARK   
2     18540093000  PL12 4EB                       STORE AND PREMISES   
3     16924543000  PL12 6AN               

In [12]:
# ==========================================
# DIAGNOSTIC: QUERY FULL ATTRITION LOG
# ==========================================
import json
import os

# Define the log path
BASE_DIR = PROJECT_ROOT
log_path = os.path.join(BASE_DIR, '05_logs/script_logs/attrition_log.json')

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        log_content = json.load(f)

    print("--- FULL ATTRITION LOG CONTENT ---")
    print(json.dumps(log_content, indent=4))
else:
    print(f"Log file not found at: {log_path}")

--- FULL ATTRITION LOG CONTENT ---
{
    "HMLR": {
        "01_Raw_National_Rows": 8043771,
        "02_Devon_Filtered_Rows": 193671,
        "03_HPI_Matched_Rows": 193102,
        "03a_Category_A_Filtered_Rows": 165929,
        "03b_Outlier_Filtered_Rows": 165614,
        "04_De-duplicated_Rows": 165614
    },
    "VOA_2017": {
        "03_Final_Devon_Rows": 63169
    },
    "VOA_2023": {
        "03_Final_Devon_Rows": 61636
    },
    "EPC": {},
    "VOA_Panel": {
        "04_Initial_Stacked_Panel": 124805,
        "05_Zero_Filtered_Panel": 124797,
        "Note": "Combined 2017 and 2023 Devon commercial assessments (zero-values removed)"
    },
    "Phase_2A_EPC_Bridge": {
        "01_Input_Residential_Rows": 165614,
        "02_Matched_UPRN_Rows": 124065,
        "03_Match_Rate_Percent": 74.91,
        "04_Property_Type_Breakdown": {
            "D": "38430 / 50403 (76.2%)",
            "F": "7747 / 25899 (29.9%)",
            "S": "32969 / 37848 (87.1%)",
            "T": "44919 /